# Joinville — build the combined hourly observations file
### `joinville_hourly_obs.csv` — one tidy table with every station's hourly record

**What this notebook does.** It merges the per-station hourly master files (one CSV per weather /
river station) into a **single long-format table**, adds each station's metadata (name, type,
coordinates, elevation), verifies that nothing was lost, and writes `joinville_hourly_obs.csv`.

You can re-run it any time your station data updates — it always rebuilds the combined file from
whatever station CSVs are present in `data/hourly/`.

---
*Author: LaCiA · PPGEC · UDESC–CCT, Joinville. Built with Claude (Cowork). Standing rule of the
project: nothing is invented — every value traces back to a measurement or a documented method.*

## 1 · What the output looks like

`joinville_hourly_obs.csv` is **"long" (tidy) format**: one row per **station × hour**, so all
stations sit in the same table and you filter with `df[df.station == "ceasa"]`.

| station | station_name | type | … | date | time_utc | temp | umid | prec | … |
|---|---|---|---|---|---|---|---|---|---|
| ceasa | Ceasa | meteorologica | … | 2026-07-14 13:00:00 | 2026-07-14 16:00:00 | 21.3 | 88 | 0.0 | … |

Rows are sorted by `station`, then `date`. A blank cell means that station does **not** measure that
variable (e.g. the rain-gauge/river stations have no temperature).

## 2 · Inputs (where the data comes from)

1. **`data/hourly/*.csv`** — the **per-station hourly masters**. Each file is named by the station
   *code* (e.g. `ceasa.csv`, `iateclube.csv`) and holds that station's full hourly history. These are
   produced/refreshed by the project's ingestion pipeline (`build_hourly_daily.py` /
   `update_datasets.py`), with rain quality-control already applied to the `prec` column
   (`rain_qc.py`).
2. **`data/geo/stations_master.csv`** — the **station registry**: `code, name, type, lat, lon,
   elevation, …`. Used to attach a human-readable name, the station type, and the coordinates to each
   row.

> The notebook combines **every** `.csv` it finds in `data/hourly/`. Add or remove a station there and
> the next run reflects it automatically. (Note: UDESC's hourly master is maintained separately from
> the weekly pipeline — keep `data/hourly/udesc.csv` up to date if you want it included.)

## 2·5 · How the weekly `.dat` files fit in (the full flow)

The raw data arrives as **Campbell logger files (`.dat`, TOA5 format)** — typically `*_HR.dat` for the
hourly table. **This notebook does not read `.dat` directly.** They are first turned into the
per-station CSV masters by the project's ingestion pipeline, and only then combined here:

```
 data/incoming/  *_HR.dat        raw Campbell logger (hourly table, TOA5)
       │   update_datasets.py   → parses TOA5 (toa5.py), de-duplicates, appends;
       │                          rain_qc.py cleans the `prec` column
       ▼
 data/hourly/  <code>.csv        per-station hourly MASTER (grows each week)
       │   ► THIS NOTEBOOK ◄     combine + attach metadata + verify
       ▼
 data/joinville_hourly_obs.csv   one tidy table, all stations
```

**Weekly routine:** (1) drop the new `.dat` files in `data/incoming/` and run the ingestion pipeline
— or just push them to trigger the GitHub Action — which refreshes `data/hourly/*.csv`; (2) run this
notebook (Restart & Run All) to rebuild the combined file. Keeping the `.dat → master` parsing in one
place (`toa5.py` / `update_datasets.py`) means this notebook only ever consumes the finished, QC'd
CSV masters — it stays simple and always matches your pipeline.

## 3 · Station catalog (as of this build)

| code | name | type | variables |
|---|---|---|---|
| `ceasa` | Ceasa | meteorológica | temp, umid, wind, rain, … |
| `aguasdejoi` | Cia Águas de Joinville (Bucarein) | hidrometeorológica | + river level |
| `cubatao` | Cubatão | hidrometeorológica | + river level |
| `flotflux` | Cachoeira Área Central | hidrometeorológica | + river level |
| `iateclube` | Joinville Iate Club | hidrometeorológica | + river level |
| `itaum` | Itaum | meteorológica | temp, umid, wind, rain, … |
| `rodovia` | Rodovia do Arroz | meteorológica | temp, umid, wind, rain, … |
| `udesc` | UDESC (CCT) | meteorológica | temp, umid, wind, rain, … |
| `divobras` | Unidade de Obras | hidrológica | rain + level only |
| `guanabara` | Guanabara | hidrológica | rain + level only |
| `jardimparaiso` | Paraíso | hidrológica | rain + level only |

**Types:** *meteorológica* = full weather station · *hidrometeorológica* = weather **and** river
level · *hidrológica* = rain gauge / river level only (no temperature/wind).

## 4 · Column dictionary & units

| column | meaning | unit |
|---|---|---|
| `station` | station code (matches the CSV file name) | — |
| `station_name` | full station name | — |
| `type` | meteorologica / hidrometeorologica / hidrologica | — |
| `lat`, `lon` | latitude, longitude (WGS84) | decimal degrees |
| `elevation` | station elevation | m |
| `date` | timestamp — **local time** (America/São_Paulo, UTC−3) | hourly |
| `time_utc` | the same instant in **UTC** ( = `date` + 3 h ) | hourly |
| `temp` | air temperature | °C |
| `umid` | relative humidity | % |
| `prec` | precipitation accumulated in the hour (**rain-QC applied**) | mm |
| `ws` | wind speed | m/s |
| `wd` | wind direction (meteorological, 0–360) | degrees |
| `gust` | wind gust | m/s |
| `gust_dir` | gust direction | degrees |
| `solar` | solar radiation (as recorded by the station) | station units |
| `pressure` | atmospheric pressure (where available) | hPa |
| `dewpoint` | dew point | °C |
| `heat_index` | heat index / apparent temperature | °C |
| `wind_chill` | wind chill | °C |
| `level`, `level_max`, `level_min` | river stage/level and hourly extremes (hydro stations) | m |

**Conventions.** The `date` column is **local time (UTC−3)** — the same convention the dashboard
displays (verified from the solar-radiation daily cycle, which peaks at local noon). The `time_utc`
column carries the same instants in **UTC** (`date` + 3 h), for interoperability — e.g. lining up with
the WRF forecast, which is in UTC. Blank cells mean the variable is **not measured** at
that station (or that hour). `prec` has the dashboard's quality control applied — logger fault codes
(inch-value sentinels), physically impossible spikes, and post-gap "catch-up" dumps are flagged and
set to `NaN` rather than deleted (see `scripts/rain_qc.py`).

## 5 · Requirements & how to run

- **Python 3** with **pandas** (already in your Anaconda environment).
- Put this notebook anywhere inside the project (e.g. `notebooks/`). It finds the project root
  automatically by looking upward for a `data/hourly/` folder.
- **Run every cell top to bottom** (Kernel → Restart & Run All). The combined file is written to
  `data/joinville_hourly_obs.csv` and a compressed `.gz` next to it.

If auto-detection fails (unusual folder layout), set `REPO_ROOT` by hand in the config cell.

In [ ]:
# --- imports ---
import sys, gzip, shutil
from pathlib import Path
import pandas as pd

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)

In [ ]:
# --- configuration: find the project root, set input/output paths ---
def find_repo_root(start=None):
    """Walk upward from `start` (default: current folder) until we find data/hourly/."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "hourly").is_dir():
            return cand
    raise FileNotFoundError(
        "Could not locate a 'data/hourly/' folder above this notebook.\n"
        "Set REPO_ROOT manually, e.g. REPO_ROOT = Path(r'C:/.../dashboard/joinville_meteo')."
    )

REPO_ROOT  = find_repo_root()                       # <-- or hard-code a Path here
INPUT_DIR  = REPO_ROOT / "data" / "hourly"          # per-station hourly masters
META_CSV   = REPO_ROOT / "data" / "geo" / "stations_master.csv"
OUTPUT_CSV = REPO_ROOT / "data" / "joinville_hourly_obs.csv"
MAKE_GZIP  = True                                   # also write a compressed .csv.gz (much smaller)

print("Project root :", REPO_ROOT)
print("Input dir    :", INPUT_DIR)
print("Metadata     :", META_CSV)
print("Output       :", OUTPUT_CSV)

In [ ]:
# --- 1) discover the per-station files ---
station_files = sorted(INPUT_DIR.glob("*.csv"))
assert station_files, f"No station CSVs found in {INPUT_DIR}"
print(f"Found {len(station_files)} station files:")
for f in station_files:
    print("  ", f.stem)

In [ ]:
# --- 2) load the station registry (metadata) ---
meta = pd.read_csv(META_CSV).set_index("code")
print(f"Registry has {len(meta)} stations. Columns: {list(meta.columns)}")
meta[["name", "type", "lat", "lon", "elevation"]].head(20)

In [ ]:
# --- 3) load and tag each station ---
# Read every station file, add a `station` column (its code), and remember the row count so we can
# verify nothing is dropped when we concatenate.
parts, source_rows = [], {}
for f in station_files:
    code_ = f.stem
    d = pd.read_csv(f, low_memory=False)
    d.insert(0, "station", code_)
    source_rows[code_] = len(d)
    parts.append(d)
    print(f"  {code_:16s} {len(d):>7,} rows   cols={list(d.columns[1:])}")
print(f"\nTotal rows across sources: {sum(source_rows.values()):,}")

In [ ]:
# --- 4) concatenate (union of columns; missing variables become NaN) ---
combined = pd.concat(parts, ignore_index=True, sort=False)
print("Concatenated shape:", combined.shape)

In [ ]:
# --- 5) attach station metadata (name, type, coordinates, elevation) ---
for col in ["name", "type", "lat", "lon", "elevation"]:
    combined[f"__{col}"] = combined["station"].map(meta[col])
combined = combined.rename(columns={
    "__name": "station_name", "__type": "type",
    "__lat": "lat", "__lon": "lon", "__elevation": "elevation",
})

missing_meta = sorted(set(combined.loc[combined["station_name"].isna(), "station"]))
if missing_meta:
    print("WARNING — no registry entry for:", missing_meta, "(name/coords will be blank)")
else:
    print("All stations matched the registry. ✓")

In [ ]:
# --- 6) add a UTC timestamp, order the columns, sort the rows ---
# `date` is LOCAL time (America/São_Paulo, UTC-3); UTC = local + 3 h.
combined["time_utc"] = (pd.to_datetime(combined["date"], errors="coerce")
                        + pd.Timedelta(hours=3)).dt.strftime("%Y-%m-%d %H:%M:%S")
preferred = ["station", "station_name", "type", "lat", "lon", "elevation", "date", "time_utc",
             "temp", "umid", "prec", "ws", "wd", "gust", "gust_dir", "solar", "pressure",
             "dewpoint", "heat_index", "wind_chill", "level", "level_max", "level_min"]
cols = [c for c in preferred if c in combined.columns] + \
       [c for c in combined.columns if c not in preferred]
combined = combined[cols].sort_values(["station", "date"]).reset_index(drop=True)
combined.head(3)

In [ ]:
# --- 7) VERIFY: no data lost, everything consistent ---
# (a) row counts per station must equal the source files exactly
got = combined.groupby("station").size().to_dict()
mismatch = {k: (source_rows[k], got.get(k)) for k in source_rows if got.get(k) != source_rows[k]}
assert not mismatch, f"Row-count mismatch (source, combined): {mismatch}"

# (b) every station is present
assert combined["station"].nunique() == len(station_files)

# (c) no accidental all-empty measurement rows (a row must have at least one value beyond metadata)
meta_cols = ["station", "station_name", "type", "lat", "lon", "elevation", "date"]
value_cols = [c for c in combined.columns if c not in meta_cols]
empty_rows = combined[value_cols].isna().all(axis=1).sum()

print(f"Stations           : {combined['station'].nunique()}")
print(f"Total rows         : {len(combined):,}")
print(f"Per-station counts : match sources ✓")
print(f"Rows with no measurement at all : {empty_rows:,}")
print(f"Date range         : {combined['date'].min()}  ..  {combined['date'].max()}")

In [ ]:
# --- 8) coverage summary (rows and time span per station) ---
summary = (combined.groupby(["station", "station_name", "type"])
                   .agg(rows=("date", "size"), first=("date", "min"), last=("date", "max"))
                   .reset_index())
summary

In [ ]:
# --- 9) write the combined file (+ compressed copy) ---
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_CSV, index=False)
size_mb = OUTPUT_CSV.stat().st_size / 1e6
print(f"Wrote {OUTPUT_CSV}  ({size_mb:.1f} MB)")

if MAKE_GZIP:
    gz = OUTPUT_CSV.with_suffix(".csv.gz")
    with open(OUTPUT_CSV, "rb") as fi, gzip.open(gz, "wb", compresslevel=9) as fo:
        shutil.copyfileobj(fi, fo)
    print(f"Wrote {gz}  ({gz.stat().st_size/1e6:.1f} MB compressed)")

## 6 · Reproducibility & maintenance

- **Weekly refresh:** your ingestion pipeline updates the per-station files in `data/hourly/`. After
  it runs, just **Restart & Run All** here to rebuild `joinville_hourly_obs.csv` with the new data.
- **Adding a station:** drop its `<code>.csv` into `data/hourly/` and add a matching row to
  `data/geo/stations_master.csv`. No code change needed.
- **UDESC note:** UDESC's hourly master is not produced by the weekly pipeline, so its
  `data/hourly/udesc.csv` is maintained separately — keep it current if you want UDESC in the output.
- **Integrity:** cell 7 will **stop with an error** if any station's rows don't survive the merge, so
  a green run guarantees the combined file equals the sum of its parts.
- **Provenance:** timestamps local (UTC−3); `prec` carries the project's rain QC (`scripts/rain_qc.py`);
  station names/coordinates come from `data/geo/stations_master.csv`.

---
## 7 · Tutorial — from raw sensors to a clean dataset

### 7.1 · In plain language

Picture **11 little weather stations** scattered around Joinville. Day and night, for over a decade,
each one quietly records the **temperature, rainfall, wind and river level** every few minutes. Each
station stores its notes in its own logger, in a raw machine format (**`.dat`**) that is awkward to
read and has the occasional glitch — a stuck sensor, or a rain gauge that "reports" 254 mm in a single
hour because of an electronics fault.

On their own, those 11 piles of raw notes are hard to use. The pipeline — and this notebook as its
last step — turns them into **one clean, friendly table**: every station, every hour, side by side,
with the obvious errors already flagged and set aside, and each station's name and location attached.

The result, **`joinville_hourly_obs.csv`**, is something you can open in Excel or load in Python and
immediately ask questions like *"how much did it rain at Cubatão in March 2020?"* — without ever
wrestling with a raw logger file.

### 7.2 · The data journey (raw → processed)

The diagram below is the whole lineage: from the sensor in the field to the tidy CSV. Nothing skips a
step, and each step is a small, checkable transformation.

In [ ]:
# --- draw the raw -> processed data-flow diagram ---
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
FIGS = REPO_ROOT / "data" / "figs"; FIGS.mkdir(parents=True, exist_ok=True)

PHASES = {  # name -> edge colour
    "RAW · bruto":               "#9aa0a6",
    "PROCESSING · processamento":"#c8922e",
    "PROCESSED · processado":    "#4078f2",
    "USE · uso":                 "#50a14f",
}
FILL = {"#9aa0a6": "#f3f4f6", "#c8922e": "#fdf1dc", "#4078f2": "#e7effd", "#50a14f": "#e9f6ee"}
STEPS = [
 ("Automatic weather stations — 11 sites\nsensors: temperature, rain, wind, river level…", "RAW · bruto"),
 ("Campbell logger files · .dat (TOA5)\n*_5.dat (5-min) · *_HR.dat (hourly) · *_DIARIA.dat (daily)", "RAW · bruto"),
 ("Ingestion — update_datasets.py\nparse TOA5 (toa5.py) · de-duplicate · append", "PROCESSING · processamento"),
 ("Quality control — rain_qc.py\ninch-code sentinels · physical ceilings · gap-dumps → NaN (flagged, not deleted)", "PROCESSING · processamento"),
 ("Per-station HOURLY MASTERS\ndata/hourly/<code>.csv", "PROCESSED · processado"),
 ("THIS NOTEBOOK\ncombine · attach metadata · verify (row-count conservation)", "PROCESSED · processado"),
 ("joinville_hourly_obs.csv\none tidy table — all stations, all hours", "PROCESSED · processado"),
 ("Uses: live dashboard · analysis & statistics · sharing & citation", "USE · uso"),
]
fig, ax = plt.subplots(figsize=(9.6, 11)); ax.set_xlim(-1.3, 10.3); ax.set_ylim(0, 100); ax.axis("off")
n = len(STEPS); top = 95; gap = (top - 5) / n; boxw = 8.4; boxh = gap * 0.60; cx = 5.6
ys = []
for i, (txt, ph) in enumerate(STEPS):
    y = top - i * gap; ys.append(y); edge = PHASES[ph]
    ax.add_patch(FancyBboxPatch((cx - boxw/2, y - boxh/2), boxw, boxh,
                 boxstyle="round,pad=0.25", linewidth=1.6, edgecolor=edge, facecolor=FILL[edge]))
    ax.text(cx, y, txt, ha="center", va="center", fontsize=9.2, color="#22262b", linespacing=1.4)
    if i > 0:
        ax.annotate("", xy=(cx, ys[i] + boxh/2), xytext=(cx, ys[i-1] - boxh/2),
                    arrowprops=dict(arrowstyle="-|>", color="#6b7280", lw=1.8))
# phase brackets on the left
left = cx - boxw/2 - 0.55
for ph, edge in PHASES.items():
    idx = [k for k,(t,p) in enumerate(STEPS) if p == ph]
    ytop, ybot = ys[idx[0]] + boxh/2, ys[idx[-1]] - boxh/2
    ax.plot([left, left], [ybot, ytop], color=edge, lw=3, solid_capstyle="round")
    ax.text(left - 0.22, (ytop + ybot)/2, ph, rotation=90, ha="right", va="center",
            fontsize=8.3, color=edge, fontweight="bold")
ax.set_title("From raw sensor logs to a clean, analysis-ready dataset", fontsize=13, fontweight="bold", pad=10)
ax.text(cx, 1.5, "Joinville hydrometeorological network · LaCiA · PPGEC · UDESC–CCT", ha="center", fontsize=8, color="#9aa0a6")
plt.tight_layout(); plt.savefig(FIGS / "data_flow.png", dpi=150, bbox_inches="tight"); plt.show()

### 7.3 · Reproducible **and** replicable — what that means here

These two words are often mixed up. They are different promises, and this workflow is built to keep
both.

**Reproducible** = *same data + same code → the same result, every time.*
- **Deterministic:** the notebook uses no randomness, no wall-clock logic, and makes no network calls,
  so running it twice produces byte-identical output.
- **Pinned inputs:** it reads explicit files under `data/hourly/` plus the station registry, and prints
  exactly which files it used (cell 1).
- **Self-checking:** it verifies **row-count conservation** and stops with an error if any station's
  rows are lost in the merge (cell 7) — so a green run *proves* the output equals the sum of its parts.

**Replicable** = *an independent person, following the documented method (even with new data), reaches
consistent results.*
- **Open method & code:** the raw format (Campbell **TOA5**), the ingestion (`toa5.py` /
  `update_datasets.py`) and the quality-control rules (`rain_qc.py`) are documented and versioned;
  anyone can apply the same procedure to their own logger network.
- **Documented, principled QC:** quality control follows **physical-plausibility** screening with
  *flag-don't-delete* and an audit trail (`data/processed/rain_qc_flags.csv`), consistent with the
  **WMO Guide to Instruments and Methods of Observation (WMO‑No. 8)**. Suspect values become `NaN`, never
  silent zeros, so every decision is transparent and re-checkable.
- **Portable outputs:** a plain **long-format CSV** with a documented column dictionary (§4) and a
  fixed timezone convention (local, UTC−3) — no proprietary format, loads anywhere.

> In short: the diagram in §7.2 is the *method*; the checks in cell 7 are the *proof*; and the open
> scripts + this documentation are what let someone else **repeat it on their own network**.

---
## 8 · Data coverage by station

How complete is each station's record? Three views, computed from the combined table you just built:

1. **Coverage bar** — within each station's active period, what fraction of the hourly slots actually
   have data (internal completeness, gaps and all).
2. **Availability over time** — a station × year grid showing *when* each station was recording.
3. **Variable completeness** — a station × variable grid showing *what* each station measures and how
   completely.

In [ ]:
# --- compute coverage metrics from `combined` ---
import numpy as np, calendar
from matplotlib.colors import LinearSegmentedColormap
BLUES = LinearSegmentedColormap.from_list("brandblue", ["#eef3fe", "#9cbcf6", "#4078f2", "#17356f"])
TYPE_COLORS = {"meteorologica": "#4078f2", "hidrometeorologica": "#e69f00", "hidrologica": "#cc79a7"}

dt = pd.to_datetime(combined["date"], errors="coerce")
work = combined.assign(_h=dt.dt.floor("h"), _year=dt.dt.year)

rows = []
for st, g in work.groupby("station"):
    h = g["_h"].dropna()
    present = h.nunique()
    span = int((h.max() - h.min()).total_seconds() // 3600) + 1        # hourly slots in the active span
    rows.append(dict(station=st,
                     name=meta.loc[st, "name"] if st in meta.index else st,
                     type=meta.loc[st, "type"] if st in meta.index else "?",
                     coverage=100 * present / span,
                     first=h.min().date(), last=h.max().date(),
                     present=present, span=span))
cov = pd.DataFrame(rows).sort_values("coverage")
cov[["name", "type", "coverage", "first", "last", "present", "span"]].round(1)

In [ ]:
# --- Plot 1: coverage (%) by station, coloured by station type ---
fig, ax = plt.subplots(figsize=(9.2, 5.4))
colors = [TYPE_COLORS.get(t, "#9aa0a6") for t in cov["type"]]
bars = ax.barh(cov["name"], cov["coverage"], color=colors, height=0.72)
for b, v, f, l in zip(bars, cov["coverage"], cov["first"], cov["last"]):
    ax.text(v + 1.2, b.get_y() + b.get_height()/2, f"{v:.0f}%  ({f.year}–{l.year})",
            va="center", fontsize=8.6, color="#22262b", fontweight="bold")
ax.set_xlim(0, 118); ax.set_xlabel("Hourly completeness within the station's active period (%)")
ax.set_title("Coverage by station — completeness while the station was active", fontsize=12, fontweight="bold")
ax.grid(axis="x", color="#eef0f2"); ax.set_axisbelow(True); ax.tick_params(length=0)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
ax.legend(handles=[Patch(color=c, label=t) for t, c in TYPE_COLORS.items()],
          title="station type", loc="lower right", frameon=False, fontsize=8.5)
plt.tight_layout(); plt.savefig(FIGS / "coverage_by_station.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# --- Plot 2: temporal availability — % of each year with hourly data (station x year) ---
hpy = work.dropna(subset=["_h"]).groupby(["station", "_year"])["_h"].nunique().unstack(fill_value=0)
exp = pd.Series({y: (8784 if calendar.isleap(int(y)) else 8760) for y in hpy.columns})
pct_year = (hpy.div(exp, axis=1) * 100).clip(upper=100)
order = cov.sort_values("first")["station"]; pct_year = pct_year.loc[order]
names = [meta.loc[s, "name"] if s in meta.index else s for s in pct_year.index]

fig, ax = plt.subplots(figsize=(min(2 + 0.62 * len(pct_year.columns), 13), 4.8))
im = ax.imshow(pct_year.values, aspect="auto", cmap=BLUES, vmin=0, vmax=100)
ax.set_xticks(range(len(pct_year.columns))); ax.set_xticklabels([int(y) for y in pct_year.columns], fontsize=8)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8.6)
ax.set_title("Temporal availability — % of each year with hourly data", fontsize=12, fontweight="bold")
ax.set_xticks(np.arange(-.5, len(pct_year.columns), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(names), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.2); ax.tick_params(which="both", length=0)
cbar = fig.colorbar(im, ax=ax, fraction=0.022, pad=0.01); cbar.set_label("% of year", fontsize=8)
plt.tight_layout(); plt.savefig(FIGS / "coverage_by_year.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# --- Plot 3: variable completeness — % of a station's hours that carry each variable ---
value_cols = [c for c in ["temp","umid","prec","ws","wd","gust","gust_dir","solar","pressure",
                          "dewpoint","heat_index","wind_chill","level","level_max","level_min"]
              if c in combined.columns]
comp = combined.groupby("station")[value_cols].apply(lambda d: d.notna().mean() * 100)
comp = comp.loc[cov.sort_values(["type", "name"])["station"]]
names = [meta.loc[s, "name"] if s in meta.index else s for s in comp.index]

fig, ax = plt.subplots(figsize=(1.6 + 0.64 * len(value_cols), 0.5 * len(comp) + 1.6))
im = ax.imshow(comp.values, aspect="auto", cmap=BLUES, vmin=0, vmax=100)
ax.set_xticks(range(len(value_cols))); ax.set_xticklabels(value_cols, rotation=45, ha="right", fontsize=8.4)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8.6)
for i in range(comp.shape[0]):
    for j in range(comp.shape[1]):
        v = comp.values[i, j]
        if v > 0:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6.8,
                    color="white" if v > 55 else "#22262b")
ax.set_title("Variable completeness — % of hours with a value (station × variable)", fontsize=12, fontweight="bold")
ax.set_xticks(np.arange(-.5, len(value_cols), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(names), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.2); ax.tick_params(which="both", length=0)
cbar = fig.colorbar(im, ax=ax, fraction=0.022, pad=0.01); cbar.set_label("% non-missing", fontsize=8)
plt.tight_layout(); plt.savefig(FIGS / "completeness_by_variable.png", dpi=150, bbox_inches="tight"); plt.show()

**How to read these.** A meteorological station running continuously should be a **solid dark row**
in plots 2–3. Pale cells flag **gaps** (a sensor down, a station retired) or **variables the station
does not measure** (e.g. the hidrológicas have only rain and river level). The `prec` column is rarely
a perfect 100% because the rain QC deliberately blanks out logger-fault values — that is the QC working,
not missing data. The four figures are also saved as PNGs in `data/figs/` for reports.